# Fresh Food Access Dashboard Build Notebook

This notebook prepares a tract-level dataset for the Atlanta fresh food access dashboard.


## 1. Imports


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

## 2. Paths


In [2]:
ROOT = Path.cwd().resolve().parent
DATA = ROOT / 'data'

ATL_PATH = DATA / 'atl' / 'Census2020_Tracts_COA.geojson'
LILA_PATH = DATA / 'lila' / 'lila_halfmi_census.shp'
BUS_PATH = DATA / 'bus_lic' / 'bus_lic.xlsx'
SNAP_HOUSEHOLDS_PATH = DATA / 'snap' / 'snap-households-1.xlsx'
SNAP_PERSONS_PATH = DATA / 'snap' / 'snap-persons-1.xlsx'
SNAP_BENEFITS_PATH = DATA / 'snap' / 'snap-benefits-1.xlsx'
MARTA_ROUTES_PATH = DATA / 'marta' / 'MARTA_Routes.geojson'

OUTPUT_PATH = ROOT / 'data' / 'processed' / 'tract_level_food_access.geojson'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


## 3. Load core tract geometry


In [3]:
atl = gpd.read_file(ATL_PATH)
atl = atl.to_crs(epsg=4326)
atl.head()


,FID,GEOID,NAME,County_Nam,State_Name,P0010001,Longitude,Latitude,Shape__Are,Shape__Len,NPU,Shape__Area,Shape__Length,geometry
0,1,13121010307,Census Tract 103.07,Fulton County,Georgia,5425,-9.411489e+06,3.980673e+06,1.534330e+07,19185.041477,,1.534330e+07,19185.041477,"POLYGON ((-84.5659 33.64711, -84.56586 33.6472..."
1,2,13121011306,Census Tract 113.06,Fulton County,Georgia,3371,-9.406914e+06,3.980658e+06,1.038947e+07,16111.869887,,1.038947e+07,16111.869887,"POLYGON ((-84.52162 33.65584, -84.52143 33.655..."
2,3,13121006000,Census Tract 60,Fulton County,Georgia,3140,-9.399711e+06,3.993669e+06,2.270683e+06,7679.561985,,2.270683e+06,7679.561985,"POLYGON ((-84.44736 33.73455, -84.4472 33.7354..."
3,4,13121006100,Census Tract 61,Fulton County,Georgia,3269,-9.399488e+06,3.992305e+06,3.256851e+06,8565.052953,,3.256851e+06,8565.052953,"POLYGON ((-84.45043 33.72352, -84.4502 33.7242..."
4,5,13121007602,Census Tract 76.02,Fulton County,Georgia,2309,-9.402284e+06,3.990050e+06,4.882680e+06,12249.788746,,4.882680e+06,12249.788746,"POLYGON ((-84.47687 33.70507, -84.47685 33.706..."


## 4. Inspect tract columns
Run this first and decide which tract ID column you want to use.


In [4]:
print(atl.columns.tolist())


['FID', 'GEOID', 'NAME', 'County_Nam', 'State_Name', 'P0010001', 'Longitude', 'Latitude', 'Shape__Are', 'Shape__Len', 'NPU', 'Shape__Area', 'Shape__Length', 'geometry']


## 5. Create master tract dataframe
Replace `TRACTCE` below if another tract key makes more sense.


In [5]:
tracts = atl.copy()
tracts['tract_id'] = tracts['NAME'].astype(str).str.zfill(6)
tracts = tracts[['tract_id', 'geometry'] + [c for c in tracts.columns if c not in ['tract_id', 'geometry']]]
tracts.head()


,tract_id,geometry,FID,GEOID,NAME,County_Nam,State_Name,P0010001,Longitude,Latitude,Shape__Are,Shape__Len,NPU,Shape__Area,Shape__Length
0,Census Tract 103.07,"POLYGON ((-84.5659 33.64711, -84.56586 33.6472...",1,13121010307,Census Tract 103.07,Fulton County,Georgia,5425,-9.411489e+06,3.980673e+06,1.534330e+07,19185.041477,,1.534330e+07,19185.041477
1,Census Tract 113.06,"POLYGON ((-84.52162 33.65584, -84.52143 33.655...",2,13121011306,Census Tract 113.06,Fulton County,Georgia,3371,-9.406914e+06,3.980658e+06,1.038947e+07,16111.869887,,1.038947e+07,16111.869887
2,Census Tract 60,"POLYGON ((-84.44736 33.73455, -84.4472 33.7354...",3,13121006000,Census Tract 60,Fulton County,Georgia,3140,-9.399711e+06,3.993669e+06,2.270683e+06,7679.561985,,2.270683e+06,7679.561985
3,Census Tract 61,"POLYGON ((-84.45043 33.72352, -84.4502 33.7242...",4,13121006100,Census Tract 61,Fulton County,Georgia,3269,-9.399488e+06,3.992305e+06,3.256851e+06,8565.052953,,3.256851e+06,8565.052953
4,Census Tract 76.02,"POLYGON ((-84.47687 33.70507, -84.47685 33.706...",5,13121007602,Census Tract 76.02,Fulton County,Georgia,2309,-9.402284e+06,3.990050e+06,4.882680e+06,12249.788746,,4.882680e+06,12249.788746


## 6. Add LILA indicators


In [6]:
lila = gpd.read_file(LILA_PATH).to_crs(epsg=4326)
lila_join = gpd.sjoin(tracts[['tract_id', 'geometry']], lila, how='left', predicate='intersects')

lila_summary = (
    lila_join.groupby('tract_id')
    .size()
    .reset_index(name='lila_count')
)

tracts = tracts.merge(lila_summary, on='tract_id', how='left')
tracts['lila_count'] = tracts['lila_count'].fillna(0)
tracts['has_lila'] = (tracts['lila_count'] > 0).astype(int)
tracts[['tract_id', 'lila_count', 'has_lila']].head()


,tract_id,lila_count,has_lila
0,Census Tract 103.07,10,1
1,Census Tract 113.06,101,1
2,Census Tract 60,7,1
3,Census Tract 61,56,1
4,Census Tract 76.02,48,1


## 7. Load SNAP data
You will likely need to inspect and rename columns to match your actual files.


In [ ]:
import os
print(os.getcwd())
snap_households = pd.read_excel('../data/snap/snap-households-1.xlsx')
#snap_persons = pd.read_excel('snap-persons-1.xlsx')
#snap_benefits = pd.read_excel('snap-benefits-1.xlsx')

#print('SNAP households columns:', snap_households.columns.tolist())
#print('SNAP persons columns:', snap_persons.columns.tolist())
#print('SNAP benefits columns:', snap_benefits.columns.tolist())


/Users/savannahimani/FreshFoodAccess_Project/FreshFoodAcessDashboard-InvestATL/notebooks


FileNotFoundError: [Errno 2] No such file or directory: './snap/snap-households-1.xlsx'

## 8. Clean and merge SNAP data
Edit these column names after you inspect the files.


In [ ]:
# Example placeholders — replace with real column names
# snap_households = snap_households.rename(columns={'TRACT': 'tract_id', 'households': 'snap_households'})
# snap_persons = snap_persons.rename(columns={'TRACT': 'tract_id', 'persons': 'snap_persons'})
# snap_benefits = snap_benefits.rename(columns={'TRACT': 'tract_id', 'benefits': 'snap_benefits'})

# snap_households['tract_id'] = snap_households['tract_id'].astype(str).str.zfill(6)
# snap_persons['tract_id'] = snap_persons['tract_id'].astype(str).str.zfill(6)
# snap_benefits['tract_id'] = snap_benefits['tract_id'].astype(str).str.zfill(6)

# tracts = tracts.merge(snap_households[['tract_id', 'snap_households']], on='tract_id', how='left')
# tracts = tracts.merge(snap_persons[['tract_id', 'snap_persons']], on='tract_id', how='left')
# tracts = tracts.merge(snap_benefits[['tract_id', 'snap_benefits']], on='tract_id', how='left')


## 9. Load and inspect business license data


In [ ]:
bus = pd.read_excel(BUS_PATH)
print(bus.columns.tolist())
bus.head()


## 10. Count food-related businesses by tract
Adjust the business-type filter and coordinate column names to match your file.


In [ ]:
# Example only — edit these names
# food_keywords = ['grocery', 'supermarket', 'market', 'produce', 'food']
# bus['business_type_clean'] = bus['business_type'].astype(str).str.lower()
# food_bus = bus[bus['business_type_clean'].str.contains('|'.join(food_keywords), na=False)].copy()

# food_bus_gdf = gpd.GeoDataFrame(
#     food_bus,
#     geometry=gpd.points_from_xy(food_bus['longitude'], food_bus['latitude']),
#     crs='EPSG:4326'
# )

# bus_join = gpd.sjoin(food_bus_gdf, tracts[['tract_id', 'geometry']], how='left', predicate='within')
# business_counts = bus_join.groupby('tract_id').size().reset_index(name='food_business_count')
# tracts = tracts.merge(business_counts, on='tract_id', how='left')
# tracts['food_business_count'] = tracts['food_business_count'].fillna(0)


## 11. Add MARTA route access
This version counts how many MARTA route geometries intersect each tract.


In [ ]:
marta = gpd.read_file(MARTA_ROUTES_PATH).to_crs(epsg=4326)
marta_join = gpd.sjoin(tracts[['tract_id', 'geometry']], marta, how='left', predicate='intersects')
marta_counts = marta_join.groupby('tract_id').size().reset_index(name='marta_access_count')
tracts = tracts.merge(marta_counts, on='tract_id', how='left')
tracts['marta_access_count'] = tracts['marta_access_count'].fillna(0)
tracts[['tract_id', 'marta_access_count']].head()


## 12. Fill missing values for scoring variables


In [ ]:
for col in ['lila_count', 'has_lila', 'marta_access_count', 'food_business_count', 'snap_households', 'snap_persons', 'snap_benefits']:
    if col in tracts.columns:
        tracts[col] = tracts[col].fillna(0)


## 13. Create normalized indicators


In [ ]:
score_cols = [c for c in ['snap_households', 'snap_persons', 'snap_benefits', 'lila_count', 'food_business_count', 'marta_access_count'] if c in tracts.columns]

scaler = MinMaxScaler()
scaled = scaler.fit_transform(tracts[score_cols].fillna(0))

for i, col in enumerate(score_cols):
    tracts[f'{col}_scaled'] = scaled[:, i]

if 'food_business_count_scaled' in tracts.columns:
    tracts['food_business_gap'] = 1 - tracts['food_business_count_scaled']

if 'marta_access_count_scaled' in tracts.columns:
    tracts['transit_gap'] = 1 - tracts['marta_access_count_scaled']

tracts.head()


## 14. Build a tract need score
Adjust weights once your columns are finalized.


In [ ]:
tracts['need_score'] = (
    0.35 * tracts.get('snap_households_scaled', 0) +
    0.25 * tracts.get('lila_count_scaled', 0) +
    0.20 * tracts.get('food_business_gap', 0) +
    0.20 * tracts.get('transit_gap', 0)
)

tracts['priority_level'] = pd.qcut(
    tracts['need_score'].rank(method='first'),
    q=4,
    labels=['Low', 'Moderate', 'High', 'Very High']
)

tracts[['tract_id', 'need_score', 'priority_level']].sort_values('need_score', ascending=False).head(10)


## 15. Save processed tract dataset


In [ ]:
tracts.to_file(OUTPUT_PATH, driver='GeoJSON')
print(f'Saved to: {OUTPUT_PATH}')


## 16. Quick map check


In [ ]:
tracts.plot(column='need_score', legend=True, figsize=(10, 10))
